In [37]:
import requests
import os

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'openai/gpt-oss-20b', 'object': 'model', 'created': 1754407957, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'meta-llama/llama-prompt-guard-2-86m', 'object': 'model', 'created': 1748632165, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072}, {'id': 'groq/compound-mini', 'object': 'model', 'created': 1756949707, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'qwen/qwen3-32b', 'object': 'model', 'created': 1748396646, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 40960}, {'id': 'meta-lla

In [38]:
model_data = response.json().get('data', [])

print("Available Groq Models:")
print("---------------------")
for model in model_data:
    print(f"ID: {model.get('id')}")
    print(f"Owned By: {model.get('owned_by')}")
    print(f"Active: {model.get('active')}")
    print("---------------------")

Available Groq Models:
---------------------
ID: openai/gpt-oss-20b
Owned By: OpenAI
Active: True
---------------------
ID: meta-llama/llama-prompt-guard-2-86m
Owned By: Meta
Active: True
---------------------
ID: llama-3.1-8b-instant
Owned By: Meta
Active: True
---------------------
ID: groq/compound-mini
Owned By: Groq
Active: True
---------------------
ID: qwen/qwen3-32b
Owned By: Alibaba Cloud
Active: True
---------------------
ID: meta-llama/llama-4-scout-17b-16e-instruct
Owned By: Meta
Active: True
---------------------
ID: groq/compound
Owned By: Groq
Active: True
---------------------
ID: canopylabs/orpheus-arabic-saudi
Owned By: Canopy Labs
Active: True
---------------------
ID: allam-2-7b
Owned By: SDAIA
Active: True
---------------------
ID: canopylabs/orpheus-v1-english
Owned By: Canopy Labs
Active: True
---------------------
ID: whisper-large-v3
Owned By: OpenAI
Active: True
---------------------
ID: meta-llama/llama-prompt-guard-2-22m
Owned By: Meta
Active: True
---------

# **NEW**

In [50]:
# Install and update essential multimedia manipulation tools
!pip install requests moviepy pydantic yt-dlp

In [51]:
import os
import json
import time
import requests
import yt_dlp
from moviepy.editor import VideoFileClip
from google.colab import userdata

# Safely extract your Groq API Key from Colab Secrets environment
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Groq API Key loaded successfully!")
except Exception:
    GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
    print("⚠️ Groq API Key not found in Colab Secrets. Please paste it manually if needed.")

✅ Groq API Key loaded successfully!


In [61]:
def prepare_video_and_audio(video_input, local_video_path="input_video.mp4", local_audio_path="extracted_audio.mp3"):
    """
    Handles both YouTube URLs and local file paths.
    Downloads full video with aggressive timeout recovery and retries,
    then extracts the audio track.
    """
    if "youtube.com" in video_input or "youtu.be" in video_input:
        print("📥 YouTube URL detected. Downloading optimized video stream locally...")

        ydl_opts = {
            # Requests pre-merged single file formats or up to 1080p/720p MP4 directly
            'format': 'best[ext=mp4]/best',
            'outtmpl': local_video_path,
            'quiet': False,
            # Network stability/timeout workarounds:
            'retries': 10,                 # Retry 10 times if a chunk drops
            'fragment_retries': 10,        # Retry individual video fragments 10 times
            'socket_timeout': 30,          # Don't wait forever on a frozen channel
            'file_access_retries': 5,
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_input])

        print(f"✅ YouTube video downloaded successfully as {local_video_path}!")
        video_source = local_video_path
    else:
        print("🎬 Local video file detected.")
        video_source = video_input

    print("🎧 Extracting audio track via MoviePy...")
    video = VideoFileClip(video_source)
    video.audio.write_audiofile(local_audio_path, logger=None)
    video.close()
    print(f"✅ Audio successfully extracted to {local_audio_path}!")

    return video_source, local_audio_path

In [54]:
def transcribe_audio_free(audio_path, api_key):
    """Transcribes audio using Groq's free Whisper API with full timestamps."""
    print("🎙️ Sending audio track to Groq Whisper-large-v3...")

    url = "https://api.groq.com/openai/v1/audio/transcriptions"
    headers = {"Authorization": f"Bearer {api_key}"}

    with open(audio_path, "rb") as f:
        files = {
            "file": (os.path.basename(audio_path), f, "audio/mp3")
        }
        data = {
            "model": "whisper-large-v3",
            "response_format": "verbose_json", # Preserves exact text-time sync matrices
            "temperature": "0.0"
        }

        response = requests.post(url, headers=headers, files=files, data=data)
        if response.status_code != 200:
            print(f"❌ Groq Transcription Error: {response.text}")
            return None
        return response.json()

In [65]:
def analyze_full_transcript(full_transcript_text, api_key):
    """
    Sends the entire, condensed transcript to Groq to select
    5 distinct viral moments across the entire timeline.
    """
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    prompt = f"""
You are an expert AI video editor and social media strategist.
Analyze the entire timestamped transcript below and select exactly 5 distinct, highly engaging viral moments.

CRITICAL RULES:
1. The 5 clips MUST be spread out across the entire timeline of the video (e.g., do not pick them all from the first 5 minutes). Look for distinct topics or shifts in energy.
2. Absolute NO OVERLAPPING or duplicate context. Each clip must stand completely on its own.
3. Each individual clip MUST be between 60 and 90 seconds (1 to 1.5 minutes) long. Ensure 'end_time' minus 'start_time' is between 60.0 and 90.0.

Transcript:
{full_transcript_text}

You MUST respond ONLY with a raw JSON object matching this schema. Do not write markdown blocks or prose outside the JSON:
{{
    "clips": [
        {{
            "start_time": 12.5,
            "end_time": 82.5,
            "headline": "Viral Clip Title",
            "caption": "Engaging social media caption text",
            "b_roll_description": "A highly detailed descriptive image generation prompt representing the context of this clip"
        }}
    ]
}}
"""

    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3, # Lower temperature forces stricter adherence to rules
        "response_format": {"type": "json_object"}
    }

    print("🧠 Analyzing entire video context to extract the 5 best unique timestamps...")
    response = requests.post(url, json=payload, headers=headers)

    if response.status_code != 200:
        print(f"❌ Groq API Error Status: {response.status_code}")
        print(f"Response: {response.text}")
        return None

    try:
        result_text = response.json()["choices"][0]["message"]["content"]
        return json.loads(result_text)
    except Exception as e:
        print(f"❌ Parsing Error: {e}")
        return None


In [66]:


def process_full_transcript(transcript_json, api_key, chunk_size=None):
    """
    Condenses the entire transcript into a lightweight string format to fit
    the model's context window, bypassing the need for blind chunking.
    """
    if not transcript_json:
        print("❌ Empty or invalid transcript provided.")
        return {"clips": []}

    raw_segments = transcript_json.get("segments", [])
    if not raw_segments:
        print("❌ No structural segments found in transcript.")
        return {"clips": []}

    # Format into a hyper-dense, clean text block to save token space
    condensed_lines = []
    for seg in raw_segments:
        start = seg.get("start", 0.0)
        text = seg.get("text", "").strip()
        condensed_lines.append(f"[{start:.1f}s] {text}")

    full_transcript_text = "\n".join(condensed_lines)

    # Send the whole video transcript at once
    final_result = analyze_full_transcript(full_transcript_text, api_key)

    if final_result and "clips" in final_result:
        # Extra safety check to guarantee we only process exactly 5 clips
        return {"clips": final_result["clips"][:5]}

    return {"clips": []}

In [69]:
def extract_viral_clips(video_path, clips_data):
    """Slices the source file precisely into distinct social-ready video files."""
    if not clips_data.get("clips"):
        print("❌ Slicing routine aborted: No clips structure present.")
        return

    video = VideoFileClip(video_path)

    for idx, clip in enumerate(clips_data.get("clips", [])):
        start = clip["start_time"]
        end = clip["end_time"]

        if start >= video.duration:
            print(f"⚠️ Skipping Clip {idx+1}: Timestamp lies beyond video length.")
            continue
        end = min(end, video.duration)

        print(f"✂️ Slicing Clip {idx+1} ({start}s - {end}s): '{clip['headline']}'")

        sub_clip = video.subclip(start, end)
        output_filename = f"viral_clip_{idx+1}.mp4"

        # Save output mp4 snippet locally
        sub_clip.write_videofile(output_filename, codec="libx264", audio_codec="aac", logger=None)

        # Export text sidecars containing descriptions for metadata and platform distribution
        with open(f"clip_{idx+1}_metadata.txt", "w") as f:
            f.write(f"TITLE: {clip['headline']}\n")
            f.write(f"CAPTION:\n{clip['caption']}\n\n")
            f.write(f"B-ROLL PROMPT:\n{clip['b_roll_description']}\n")

    video.close()
    print("🎉 Slicing processing completed!")



In [70]:
def generate_free_b_roll(prompt_text, clip_index):
    """Retrieves context-aware graphic frames via Pollinations AI for B-Roll coverage."""
    print(f"🎨 Querying art models for Clip {clip_index} B-roll elements...")

    cleaned_prompt = requests.utils.quote(prompt_text)
    image_url = f"https://image.pollinations.ai/p/{cleaned_prompt}?width=1024&height=1024&model=turbo&enhance=true"

    response = requests.get(image_url)
    if response.status_code == 200:
        filename = f"broll_asset_{clip_index}.jpg"
        with open(filename, "wb") as file:
            file.write(response.content)
        print(f"💾 B-roll visualization saved: {filename}")
    else:
        print(f"❌ Could not pull media for Clip {clip_index}")

In [71]:
# Paste your YouTube link or input a local filename string (e.g. "my_raw_recording.mp4")
video_input = "https://youtu.be/MB3FK2BmGSw?si=r1XJKiPfgabAY3z9"


In [62]:
# Run step 1: Download external feeds and extract audio signals
video_file, audio_file = prepare_video_and_audio(video_input)

📥 YouTube URL detected. Downloading optimized video stream locally...
[youtube] Extracting URL: https://youtu.be/MB3FK2BmGSw?si=r1XJKiPfgabAY3z9
[youtube] MB3FK2BmGSw: Downloading webpage
[youtube] MB3FK2BmGSw: Downloading android vr player API JSON
[youtube] MB3FK2BmGSw: Downloading player 2d01abf7-main
[youtube] [jsc:deno] Solving JS challenges using deno


[info] MB3FK2BmGSw: Downloading 1 format(s): 18
[download] Destination: input_video.mp4
[download] 100% of   28.32MiB in 00:00:08 at 3.16MiB/s   
✅ YouTube video downloaded successfully as input_video.mp4!
🎧 Extracting audio track via MoviePy...
✅ Audio successfully extracted to extracted_audio.mp3!


In [72]:
# Run step 2: Generate timestamp tracking matrix
transcript_data = transcribe_audio_free(audio_file, GROQ_API_KEY)

🎙️ Sending audio track to Groq Whisper-large-v3...


In [73]:
# Run step 3, 4, 5: Run contextual mapping, slice media files, and harvest art packages


In [74]:
if transcript_data:
    final_analysis = process_full_transcript(transcript_data, GROQ_API_KEY, chunk_size=150)
    print("\n--- Extracted Clips Structural Map ---")
    print(json.dumps(final_analysis, indent=2))
else:
    print("❌ Critical Breakdown: Ensure your GROQ_API_KEY is configured accurately.")

🧠 Analyzing entire video context to extract the 5 best unique timestamps...

--- Extracted Clips Structural Map ---
{
  "clips": [
    {
      "start_time": 19.3,
      "end_time": 77.9,
      "headline": "The Rise of AI in Coding",
      "caption": "Discover how AI is transforming the coding landscape! #AI #Coding",
      "b_roll_description": "A futuristic coding environment with AI-powered tools and human coders collaborating in the background, surrounded by screens displaying complex algorithms and data structures"
    },
    {
      "start_time": 134.8,
      "end_time": 224.5,
      "headline": "Unlocking the Benefits of ICPC",
      "caption": "Want to boost your career in tech? Learn about the benefits of participating in ICPC! #ICPC #TechCareer",
      "b_roll_description": "A split-screen image with a group of students participating in ICPC on one side, and a professional coder working in a top tech company on the other, highlighting the career benefits of ICPC participation"

In [75]:
if transcript_data:
    # physical video parsing
    extract_viral_clips(video_file, final_analysis)

else:
    print("❌ Critical Breakdown: Ensure your GROQ_API_KEY is configured accurately.")

✂️ Slicing Clip 1 (19.3s - 77.9s): 'The Rise of AI in Coding'
✂️ Slicing Clip 2 (134.8s - 224.5s): 'Unlocking the Benefits of ICPC'
✂️ Slicing Clip 3 (241.9s - 321.2s): 'ICPC Rules and Eligibility'
✂️ Slicing Clip 4 (379.2s - 463.2s): 'The ICPC Registration Process'
✂️ Slicing Clip 5 (602.2s - 677.7s): 'Preparing for ICPC Success'
🎉 Slicing processing completed!


In [76]:
if transcript_data:
    # B-roll image compilation
    for i, c in enumerate(final_analysis.get("clips", [])):
        generate_free_b_roll(c["b_roll_description"], i+1)
else:
    print("❌ Critical Breakdown: Ensure your GROQ_API_KEY is configured accurately.")

🎨 Querying art models for Clip 1 B-roll elements...
💾 B-roll visualization saved: broll_asset_1.jpg
🎨 Querying art models for Clip 2 B-roll elements...
💾 B-roll visualization saved: broll_asset_2.jpg
🎨 Querying art models for Clip 3 B-roll elements...
💾 B-roll visualization saved: broll_asset_3.jpg
🎨 Querying art models for Clip 4 B-roll elements...
💾 B-roll visualization saved: broll_asset_4.jpg
🎨 Querying art models for Clip 5 B-roll elements...
💾 B-roll visualization saved: broll_asset_5.jpg


In [77]:
import os
from IPython.display import display, HTML, Image, Video

In [78]:
def display_pipeline_results():
    """
    Scans the local directory for the generated video clips, text metadata,
    and B-roll art assets, displaying them cleanly in the notebook output area.
    """
    # Dynamically determine the number of clips found based on final_analysis or files on disk
    num_clips = 5
    if 'final_analysis' in globals() and isinstance(final_analysis, dict) and "clips" in final_analysis:
        num_clips = len(final_analysis["clips"])

    if num_clips == 0:
        print("❌ No clips data found. Please run the full execution pipeline first.")
        return

    print(f"👁️ Displaying Results for {num_clips} Generated Viral Clips:\n")

    for idx in range(1, num_clips + 1):
        clip_file = f"viral_clip_{idx}.mp4"
        meta_file = f"clip_{idx}_metadata.txt"
        broll_file = f"broll_asset_{idx}.jpg"

        # 1. Display Header for the Clip
        display(HTML(f"""
        <div style="background-color: #1e1e2e; padding: 12px; border-radius: 8px; margin-top: 30px; margin-bottom: 15px;">
            <h2 style="color: #cdd6f4; margin: 0; font-family: sans-serif;">🎬 VIRAL MOMENT {idx}</h2>
        </div>
        """))

        # 2. Read and Display Title & Captions Metadata
        if os.path.exists(meta_file):
            with open(meta_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            # Parse lines nicely for an HTML box
            title = lines[0].replace("TITLE:", "").strip() if len(lines) > 0 else f"Clip {idx}"
            caption = ""
            broll_prompt = ""

            # Read the multi-line elements from file structure
            content = "".join(lines)
            if "CAPTION:" in content and "B-ROLL PROMPT:" in content:
                caption = content.split("CAPTION:")[1].split("B-ROLL PROMPT:")[0].strip()
                broll_prompt = content.split("B-ROLL PROMPT:")[1].strip()

            display(HTML(f"""
            <div style="background-color: #f8f9fa; border-left: 5px solid #89b4fa; padding: 15px; margin-bottom: 15px; border-radius: 4px;">
                <b style="color: #1e1e2e; font-size: 16px;">📌 Headline:</b> <span style="font-size: 16px;">{title}</span><br><br>
                <b style="color: #1e1e2e;">📱 Social Caption:</b> <i style="color: #4c4f69;">"{caption}"</i><br><br>
                <b style="color: #1e1e2e;">🎨 B-Roll Context Prompt:</b> <code style="background-color: #e6e9ef; padding: 2px 6px; border-radius: 3px; color: #d20f39;">{broll_prompt}</code>
            </div>
            """))
        else:
            print(f"⚠️ Metadata note file '{meta_file}' not found.")

        # 3. Present Media Objects Side-by-Side (Video & Image Assets)
        # Check if files exist before rendering players
        has_video = os.path.exists(clip_file)
        has_broll = os.path.exists(broll_file)

        if has_video or has_broll:
            # We construct a side-by-side flexbox container
            display(HTML("<div style='display: flex; flex-wrap: wrap; gap: 25px; align-items: flex-start; margin-bottom: 20px;'>"))

            if has_video:
                print(f"▶️ Sliced Video Preview ({clip_file}):")
                # embed=True enables a streaming HTML5 video render directly in Colab
                display(Video(clip_file, embed=True, width=420))

            if has_broll:
                print(f"🖼️ Generated B-Roll Art Framework ({broll_file}):")
                display(Image(filename=broll_file, width=320))

            display(HTML("</div><hr style='border: 0; border-top: 1px dashed #ccc; margin-top: 20px;'>"))
        else:
            print(f"⚠️ Media files for Clip {idx} are not ready on local disk storage.")


In [80]:
# Trigger the rendering engine
display_pipeline_results()